In [4]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --break-system-packages

Looking in indexes: https://download.pytorch.org/whl/cu118Note: you may need to restart the kernel to use updated packages.

  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.0%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.6.0%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.5.1%2Bcu118-cp312-cp312-win_amd64.whl (4.0 MB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.5.0%2Bcu118-cp312-cp312-win_amd64.whl (4.0 MB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.4.1%2Bcu118-cp312-cp312-win_amd64.whl (4.0 MB)
  Using cached ht

  You can safely remove it manually.


In [1]:
%pip install surya-ocr

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ISO-IDENTIFIER EXTRACTION

In [2]:
#iso-identifier-v1-gpu-11/03/2026
import cv2
import numpy as np
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.detection import DetectionPredictor
from surya.recognition import RecognitionPredictor
import re
from pathlib import Path
from typing import List, Set, Tuple, Optional
import torch
import os

class TargetedPatchExtractor:
    """
    Targeted extension implementation - extends only specific regions
    """

    def __init__(self, patch_folder: str, patch_width: int = 640, patch_height: int = 640):
        self.patch_folder = Path(patch_folder)
        self.patch_width = patch_width
        self.patch_height = patch_height

        # ========== GPU SETUP ==========
        # Set CUDA device (use first GPU)
        os.environ['CUDA_VISIBLE_DEVICES'] = '0'
        
        # Check CUDA availability
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            print(f"\n{'='*60}")
            print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
            print(f"  CUDA Version: {torch.version.cuda}")
            print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
            print(f"{'='*60}\n")
        else:
            self.device = torch.device("cpu")
            print("\n⚠ CUDA not available, using CPU")
            print("  To enable GPU: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118\n")
        # ===============================

        print("Initializing OCR...")
        try:
            # Try passing device to predictors
            self.foundation = FoundationPredictor(device=self.device)
            self.det_predictor = DetectionPredictor(device=self.device)
            self.rec_predictor = RecognitionPredictor(self.foundation, device=self.device)
        except TypeError:
            # If device parameter not supported, initialize without it
            # Surya should auto-detect CUDA
            print("  Note: Initializing predictors without explicit device parameter")
            self.foundation = FoundationPredictor()
            self.det_predictor = DetectionPredictor()
            self.rec_predictor = RecognitionPredictor(self.foundation)

        # Complete ISO identifier pattern
        self.complete_pattern = re.compile(
            r'(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})(?:\s*-\s*([A-Z]\d{1,3}))?',
            re.IGNORECASE
        )

        # STRICT partial patterns - MUST have dashes and match ISO structure
        # ISO format: SIZE"-CODE-NUMBER-SUFFIX
        # SIZE: digits with optional /fraction
        # CODE: 2 letters (may be misread as 0W, etc)
        # NUMBER: 4-6 digits
        # SUFFIX: letter + 1-3 digits (but may be cut off as just "D", "B", "D0", etc.)

        self.partial_patterns = [
            # Missing size: -CODE-NUMBER-SUFFIX
            re.compile(r'^-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # Missing suffix (with trailing dash): SIZE"-CODE-NUMBER-
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*$', re.I),

            # Partial suffix (no trailing dash): SIZE"-CODE-NUMBER-X or SIZE"-CODE-NUMBER-X0
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,2})$', re.I),

            # Missing size and code: -NUMBER-SUFFIX
            re.compile(r'^-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # Missing code and suffix: SIZE"-NUMBER-
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*(\d{4,6})\s*-?\s*$', re.I),

            # Missing size: CODE-NUMBER-SUFFIX (no leading dash)
            re.compile(r'^([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # Missing suffix: SIZE"-CODE-NUMBER (no trailing dash)
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})$', re.I),

            # Missing size and suffix: CODE-NUMBER
            re.compile(r'^([A-Z0-9]{2})\s*-\s*(\d{4,6})$', re.I),

            # Just number-suffix: NUMBER-SUFFIX (including partial suffix)
            re.compile(r'^(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # SIZE"-CODE only (very partial)
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})$', re.I),

            # Missing everything except number: -NUMBER
            re.compile(r'^-\s*(\d{4,6})$', re.I),
        ]

    def normalize_identifier(self, size, code, number, suffix=None):
        """Normalize and fix OCR errors"""
        code = code.upper()

        # Fix OCR errors
        if code == '0W': code = 'OW'
        elif code == '0C': code = 'OC'
        elif code == 'C0': code = 'CO'
        elif code == 'CQ': code = 'CG'
        elif code == 'NC': code = 'NG'

        if not (len(code) == 2 and code.isalpha()):
            return None

        number = number.replace('O', '0').replace('I', '1')
        if not (4 <= len(number) <= 6 and number.isdigit()):
            return None

        if suffix:
            suffix = suffix.upper().replace('O', '0').replace('I', '1')
            if not (2 <= len(suffix) <= 4 and suffix[0].isalpha() and suffix[1:].isdigit()):
                return None
            return f'{size}"-{code}-{number}-{suffix}'

        return None

    def extract_text_from_predictions(self, predictions) -> List[Tuple[str, tuple]]:
        """Extract text with bboxes from Surya predictions"""
        results = []
        for line in predictions[0].text_lines:
            text = line.text.strip()
            if text:
                # bbox format: (x_min, y_min, x_max, y_max)
                results.append((text, line.bbox))
        return results

    def extract_complete_identifiers(self, texts_with_bboxes: List[Tuple[str, tuple]]) -> Set[str]:
        """Extract complete ISO identifiers"""
        found = set()

        for text, _ in texts_with_bboxes:
            match = self.complete_pattern.search(text)
            if match:
                size = match.group(1)
                code = match.group(2)
                number = match.group(3)
                suffix = match.group(4)

                formatted = self.normalize_identifier(size, code, number, suffix)
                if formatted:
                    found.add(formatted)

        return found

    def has_partial_pattern(self, text: str) -> bool:

        text = text.strip()

        # Must contain dash
        if '-' not in text:
            return False

        # If contains quote mark ("), it's likely an ISO identifier with size
        if '"' in text:
            # Check against partial patterns
            for pattern in self.partial_patterns:
                if pattern.match(text):
                    return True
            return False

        # No quote mark - more careful checking needed

        # Reject equipment tags (2-3 letters + dash + exactly 4 digits, nothing else)
        # Example: EI-8508, FI-8530, PT-8113
        equipment_tag_pattern = re.compile(r'^[A-Z]{2,3}-\d{4}$', re.I)
        if equipment_tag_pattern.match(text):
            return False

        # Reject document numbers (contains more than 4 dash-separated parts)
        # Example: SPP-11-09-0110 (4 parts), THAI-3C-SPP-11-09-0304 (6 parts)
        parts = text.split('-')
        if len(parts) > 4:  # ISO identifiers have max 4 parts
            return False

        # Additional check: if it has exactly 4 parts and none look like ISO components, reject
        if len(parts) == 4:
            # Check if any part looks like it could be ISO number (4+ digits)
            has_long_number = any(part.isdigit() and len(part) >= 4 for part in parts)
            if not has_long_number:
                return False

        # Check against valid partial patterns
        for pattern in self.partial_patterns:
            if pattern.match(text):
                return True

        return False

    def get_patch_position(self, patch_name: str) -> Tuple[Optional[int], Optional[int]]:
        """Extract row and column from patch name"""
        match = re.search(r'_r(\d+)_c(\d+)', patch_name)
        if match:
            return int(match.group(1)), int(match.group(2))
        return None, None

    def determine_edge_and_direction(self, bbox: tuple, width: int, height: int,
                                     edge_threshold: int = 80) -> Tuple[Optional[str], Optional[str]]:

        x_min, y_min, x_max, y_max = bbox

        # Check which edge the bbox is closest to
        if y_min < edge_threshold:
            return 'top', 'up'
        elif y_max > (height - edge_threshold):
            return 'bottom', 'down'
        elif x_min < edge_threshold:
            return 'left', 'left'
        elif x_max > (width - edge_threshold):
            return 'right', 'right'

        return None, None

    def get_neighbor_patch(self, patch_name: str, direction: str) -> Optional[str]:
        """Get neighbor patch name based on direction"""
        row, col = self.get_patch_position(patch_name)
        if row is None:
            return None

        # Calculate neighbor position
        if direction == 'up':
            row -= 1
        elif direction == 'down':
            row += 1
        elif direction == 'left':
            col -= 1
        elif direction == 'right':
            col += 1

        # Check bounds (4 rows × 6 columns)
        if row < 0 or row >= 4 or col < 0 or col >= 6:
            return None

        base_name = patch_name.split('_r')[0]
        return f"{base_name}_r{row}_c{col}.png"

    def create_targeted_extension(self, current_patch: np.ndarray,
                                  neighbor_patch: np.ndarray,
                                  bbox: tuple,
                                  direction: str,
                                  padding: int = 40,
                                  lateral_margin: int = 50) -> Optional[np.ndarray]:

        # Convert bbox to integers
        x_min, y_min, x_max, y_max = map(int, bbox)
        h_curr, w_curr = current_patch.shape[:2]
        h_neigh, w_neigh = neighbor_patch.shape[:2]

        # Add lateral margins (convert to int)
        x_min_expanded = int(max(0, x_min - lateral_margin))
        x_max_expanded = int(min(w_curr, x_max + lateral_margin))
        y_min_expanded = int(max(0, y_min - lateral_margin))
        y_max_expanded = int(min(h_curr, y_max + lateral_margin))

        if direction == 'up':
            # Extend upward into neighbor's bottom region
            # Take: [bottom of neighbor] + [top portion of current around bbox]

            # From neighbor: take bottom padding pixels at same x-range
            x_min_neigh = int(max(0, x_min_expanded))
            x_max_neigh = int(min(w_neigh, x_max_expanded))
            neighbor_strip = neighbor_patch[-padding:, x_min_neigh:x_max_neigh]

            # From current: take region around bbox
            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]

            # Stack vertically: neighbor bottom + current region
            extended = np.vstack([neighbor_strip, current_region])

        elif direction == 'down':
            # Extend downward into neighbor's top region

            # From current: take region around bbox
            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]

            # From neighbor: take top padding pixels at same x-range
            x_min_neigh = int(max(0, x_min_expanded))
            x_max_neigh = int(min(w_neigh, x_max_expanded))
            neighbor_strip = neighbor_patch[:padding, x_min_neigh:x_max_neigh]

            # Stack vertically: current region + neighbor top
            extended = np.vstack([current_region, neighbor_strip])

        elif direction == 'left':
            # Extend leftward into neighbor's right region

            # From neighbor: take right padding pixels at same y-range
            y_min_neigh = int(max(0, y_min_expanded))
            y_max_neigh = int(min(h_neigh, y_max_expanded))
            neighbor_strip = neighbor_patch[y_min_neigh:y_max_neigh, -padding:]

            # From current: take region around bbox
            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]

            # Stack horizontally: neighbor right + current region
            extended = np.hstack([neighbor_strip, current_region])

        elif direction == 'right':
            # Extend rightward into neighbor's left region

            # From current: take region around bbox
            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]

            # From neighbor: take left padding pixels at same y-range
            y_min_neigh = int(max(0, y_min_expanded))
            y_max_neigh = int(min(h_neigh, y_max_expanded))
            neighbor_strip = neighbor_patch[y_min_neigh:y_max_neigh, :padding]

            # Stack horizontally: current region + neighbor left
            extended = np.hstack([current_region, neighbor_strip])

        else:
            return None

        return extended

    def run_ocr_on_image(self, img: np.ndarray) -> Tuple[Set[str], List[Tuple[str, tuple]]]:
        """Run OCR on image array"""
        # Convert BGR to RGB for PIL
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)

        # Run OCR
        predictions = self.rec_predictor([img_pil], det_predictor=self.det_predictor)

        # Extract text with bboxes
        texts_with_bboxes = self.extract_text_from_predictions(predictions)

        # Find complete identifiers
        complete_ids = self.extract_complete_identifiers(texts_with_bboxes)

        return complete_ids, texts_with_bboxes

    def process_single_patch(self, patch_path: Path) -> Tuple[Set[str], List[dict]]:
        """Process a single patch to find complete and partial identifiers"""
        print(f"\nProcessing: {patch_path.name}")

        # Load patch
        patch = cv2.imread(str(patch_path))
        if patch is None:
            print(f"  ERROR: Could not load patch")
            return set(), []

        height, width = patch.shape[:2]

        # Run OCR
        complete_ids, texts_with_bboxes = self.run_ocr_on_image(patch)

        if complete_ids:
            print(f"  ✓ Complete: {complete_ids}")

        # Debug: Show all detected text
        print(f"  📝 All detected text ({len(texts_with_bboxes)} items):")
        for text, bbox in texts_with_bboxes[:10]:  # Show first 10 only
            print(f"     '{text}'")
        if len(texts_with_bboxes) > 10:
            print(f"     ... and {len(texts_with_bboxes) - 10} more")

        # Find partials
        partials_info = []
        for text, bbox in texts_with_bboxes:
            # Skip if part of complete identifier
            if any(text in cid for cid in complete_ids):
                continue

            # Check if partial pattern
            if self.has_partial_pattern(text):
                edge, direction = self.determine_edge_and_direction(bbox, width, height)

                if edge and direction:
                    print(f"  ⚠ Partial on {edge}: '{text}' at bbox {bbox}")
                    partials_info.append({
                        'text': text,
                        'bbox': bbox,
                        'edge': edge,
                        'direction': direction
                    })
                else:
                    # Partial found but not on edge
                    print(f"  ℹ Partial (not on edge): '{text}'")

        return complete_ids, partials_info

    def process_extensions(self, patch_path: Path, partials_info: List[dict]) -> Set[str]:
        """Process targeted extensions for partial identifiers"""
        found_in_extensions = set()

        if not partials_info:
            return found_in_extensions

        # Load current patch once
        current_patch = cv2.imread(str(patch_path))
        if current_patch is None:
            return found_in_extensions

        for partial in partials_info:
            text = partial['text']
            bbox = partial['bbox']
            direction = partial['direction']

            # Get neighbor patch
            neighbor_name = self.get_neighbor_patch(patch_path.name, direction)
            if not neighbor_name:
                print(f"  ⚠ No neighbor for '{text}' in {direction} direction")
                continue

            neighbor_path = self.patch_folder / neighbor_name
            if not neighbor_path.exists():
                print(f"  ⚠ Neighbor not found: {neighbor_name}")
                continue

            # Load neighbor patch
            neighbor_patch = cv2.imread(str(neighbor_path))
            if neighbor_patch is None:
                print(f"  ⚠ Could not load neighbor: {neighbor_name}")
                continue

            print(f"  → Creating targeted extension to {neighbor_name} ({direction}) for '{text}'")

            # Create targeted extended region
            extended_region = self.create_targeted_extension(
                current_patch,
                neighbor_patch,
                bbox,
                direction,
                padding=40,
                lateral_margin=50
            )

            if extended_region is None or extended_region.size == 0:
                print(f"  ✗ Failed to create extended region")
                continue

            # Run OCR on small extended region
            try:
                complete_ids, _ = self.run_ocr_on_image(extended_region)

                if complete_ids:
                    print(f"  ✓ Found in extension: {complete_ids}")
                    found_in_extensions.update(complete_ids)
                else:
                    print(f"  ⚠ No complete identifiers in extension")

            except Exception as e:
                print(f"  ✗ OCR error: {e}")

        return found_in_extensions

    def extract_from_page(self, page_name: str) -> dict:
        """Extract all ISO identifiers from a page"""
        print("="*80)
        print(f"EXTRACTING FROM PAGE: {page_name}")
        print("="*80)

        # Find all patches
        patch_files = sorted(self.patch_folder.glob(f"{page_name}_r*_c*.png"))
        print(f"\nFound {len(patch_files)} patches")

        all_complete = set()
        all_from_extensions = set()

        # Process each patch
        for patch_path in patch_files:
            # Get complete and partials from patch
            complete_ids, partials_info = self.process_single_patch(patch_path)
            all_complete.update(complete_ids)

            # Process extensions for partials
            if partials_info:
                extension_ids = self.process_extensions(patch_path, partials_info)
                all_from_extensions.update(extension_ids)

        # Combine results
        all_identifiers = all_complete | all_from_extensions

        return {
            'page_name': page_name,
            'total_patches': len(patch_files),
            'identifiers': sorted(all_identifiers),
            'total_found': len(all_identifiers),
            'from_complete': len(all_complete),
            'from_extensions': len(all_from_extensions),
        }


def check_cuda():
    """Verify CUDA availability and GPU info"""
    print("\n" + "="*80)
    print("CUDA CHECK")
    print("="*80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("⚠ WARNING: CUDA not available - will use CPU")
        print("  Install CUDA-enabled PyTorch:")
        print("  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118")
    print("="*80 + "\n")


def main():
    # Configuration
    patch_folder = r"E:\Projects\P&ID Versions\P&ID V2\iso-identifier-test-patches\test-node1.4-patches"
    page_name = "ERCP_2"

    # Create extractor
    extractor = TargetedPatchExtractor(patch_folder)

    # Extract
    results = extractor.extract_from_page(page_name)

    # Print results
    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    print(f"\nPage: {results['page_name']}")
    print(f"Total patches: {results['total_patches']}")
    print(f"\nTotal identifiers: {results['total_found']}")
    print(f"  - From complete patches: {results['from_complete']}")
    print(f"  - From targeted extensions: {results['from_extensions']}")

    print(f"\nAll identifiers ({len(results['identifiers'])}):")
    for idx, identifier in enumerate(results['identifiers'], 1):
        print(f"  {idx:2d}. {identifier}")

    # Check critical identifiers
    print("\n" + "="*80)
    print("CHECKING CRITICAL IDENTIFIERS")
    print("="*80)

    critical = [
        "1/2\"-CP-8113-D48",
        "6\"-OW-8104-D03",
        "8\"-OW-8105-D03",
        "6\"-OW-8601-D03",
        "8\"-CL-8903-D03",
        "24\"-CG-8111",
    ]

    found_set = set(results['identifiers'])

    for c in critical:
        if c in found_set:
            print(f"  ✓ {c} - FOUND")
        else:
            # Check for similar (size might be wrong)
            similar = [v for v in found_set if c.split('-')[1:] == v.split('-')[1:] if '-' in v and '-' in c]
            if similar:
                print(f"  ~ {c} - Similar: {similar[0]}")
            else:
                print(f"  ✗ {c} - MISSING")

    print("\n" + "="*80)

    return results


if __name__ == "__main__":
    check_cuda()
    main()

C:\Users\saroc\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



CUDA CHECK
PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Memory: 4.29 GB


✓ Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
  CUDA Version: 11.8
  GPU Memory: 4.29 GB

Initializing OCR...
  Note: Initializing predictors without explicit device parameter
EXTRACTING FROM PAGE: ERCP_2

Found 24 patches

Processing: ERCP_2_r0_c0.png


Recognizing Text: 100%|██████████| 29/29 [00:02<00:00, 13.09it/s]


  ✓ Complete: {'2"-FG-8104-B03', '2"-FG-8112-B03'}
  📝 All detected text (29 items):
     'FΙ'
     '&136B'
     '300#'
     '300# 1"'
     '1”'
     '2"-FG-8112-B03'
     'NC'
     '2"x1"'
     '2"x1"'
     '300#'
     ... and 19 more

Processing: ERCP_2_r0_c1.png


Recognizing Text: 100%|██████████| 35/35 [00:01<00:00, 26.70it/s]


...

  ⚠ No complete identifiers in extension

Processing: ERCP_2_r3_c4.png


Recognizing Text: 100%|██████████| 50/50 [00:03<00:00, 15.42it/s]


  📝 All detected text (50 items):
     '. . .'
     '. 0, 0. , .'
     'REV'
     'DATE'
     'TO FY 9110 D-9110 OILY WATER'
     'FLOW CONTROL SYSTEM'
     '09'
     'SW'
     'AS BUILT FOR GBN'
     '18,08.22'
     ... and 40 more

Processing: ERCP_2_r3_c5.png


Recognizing Text: 100%|██████████| 48/48 [00:05<00:00,  8.52it/s]

  📝 All detected text (48 items):
     '. . .'
     '...'
     'DESCRIPTION'
     'RY'
     'CHKD'
     'APPR'
     'CONFIDENTIAL'
     '-MOD-19065'
     '<b>KPA</b>'
     'KW'
     ... and 38 more

FINAL RESULTS

Page: ERCP_2
Total patches: 24

Total identifiers: 26
  - From complete patches: 22
  - From targeted extensions: 5

All identifiers (26):
   1. 1/2"-CP-8113-D48
   2. 10"-CL-8111-D03
   3. 12"-HF-8544-B03
   4. 14"-HF-8554-B03
   5. 18"-HF-8526-B03
   6. 18"-NG-8113-D48
   7. 2"-CD-8351-D03
   8. 2"-CL-8903-D03
   9. 2"-CL-8993-D06
  10. 2"-FG-8104-B03
  11. 2"-FG-8112-B03
  12. 2"-OW-8112-D03
  13. 20"-HF-8517-B03
  14. 24"-CG-8111-D48
  15. 24"-NG-8114-D48
  16. 24"-NG-8831-D48
  17. 3"-CD-8310-D0
  18. 3"-CD-8310-D03
  19. 3"-NG-8980-D48
  20. 6"-OW-8104-D03
  21. 6"-OW-8601-D03
  22. 8"-CL-8601-D03
  23. 8"-FW-82001-B90
  24. 8"-NG-8116-D48
  25. 8"-OW-8104-D03
  26. 8"-OW-8105-D03

CHECKING CRITICAL IDENTIFIERS
  ✓ 1/2"-CP-8113-D48 - FOUND
  ✓ 6"-OW-8104-D03 - FOUND
  ✓

In [4]:
#iso-identifier-v2-multipage-11/03/2026
import cv2
import numpy as np
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.detection import DetectionPredictor
from surya.recognition import RecognitionPredictor
import re
from pathlib import Path
from typing import List, Set, Tuple, Optional, Dict
import torch
import os

# Fix for CUDNN_STATUS_BAD_PARAM_STREAM_MISMATCH
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

class TargetedPatchExtractor:
    """
    Targeted extension implementation - extends only specific regions
    Supports multiple pages in same folder
    """

    def __init__(self, patch_folder: str, patch_width: int = 640, patch_height: int = 640):
        self.patch_folder = Path(patch_folder)
        self.patch_width = patch_width
        self.patch_height = patch_height

        # ========== GPU SETUP ==========
        # Set CUDA device (use first GPU)
        os.environ['CUDA_VISIBLE_DEVICES'] = '0'
        
        # Check CUDA availability
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            print(f"\n{'='*60}")
            print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
            print(f"  CUDA Version: {torch.version.cuda}")
            print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
            print(f"{'='*60}\n")
        else:
            self.device = torch.device("cpu")
            print("\n⚠ CUDA not available, using CPU")
            print("  To enable GPU: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118\n")
        # ===============================

        print("Initializing OCR...")
        try:
            # Try passing device to predictors
            self.foundation = FoundationPredictor(device=self.device)
            self.det_predictor = DetectionPredictor(device=self.device)
            self.rec_predictor = RecognitionPredictor(self.foundation, device=self.device)
        except TypeError:
            # If device parameter not supported, initialize without it
            # Surya should auto-detect CUDA
            print("  Note: Initializing predictors without explicit device parameter")
            self.foundation = FoundationPredictor()
            self.det_predictor = DetectionPredictor()
            self.rec_predictor = RecognitionPredictor(self.foundation)

        # Complete ISO identifier pattern
        self.complete_pattern = re.compile(
            r'(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})(?:\s*-\s*([A-Z]\d{1,3}))?',
            re.IGNORECASE
        )

        # STRICT partial patterns - MUST have dashes and match ISO structure
        self.partial_patterns = [
            # Missing size: -CODE-NUMBER-SUFFIX
            re.compile(r'^-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # Missing suffix (with trailing dash): SIZE"-CODE-NUMBER-
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*$', re.I),

            # Partial suffix (no trailing dash): SIZE"-CODE-NUMBER-X or SIZE"-CODE-NUMBER-X0
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,2})$', re.I),

            # Missing size and code: -NUMBER-SUFFIX
            re.compile(r'^-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # Missing code and suffix: SIZE"-NUMBER-
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*(\d{4,6})\s*-?\s*$', re.I),

            # Missing size: CODE-NUMBER-SUFFIX (no leading dash)
            re.compile(r'^([A-Z0-9]{2})\s*-\s*(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # Missing suffix: SIZE"-CODE-NUMBER (no trailing dash)
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})$', re.I),

            # Missing size and suffix: CODE-NUMBER
            re.compile(r'^([A-Z0-9]{2})\s*-\s*(\d{4,6})$', re.I),

            # Just number-suffix: NUMBER-SUFFIX (including partial suffix)
            re.compile(r'^(\d{4,6})\s*-\s*([A-Z]\d{0,3})$', re.I),

            # SIZE"-CODE only (very partial)
            re.compile(r'^(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})$', re.I),

            # Missing everything except number: -NUMBER
            re.compile(r'^-\s*(\d{4,6})$', re.I),
        ]

    def discover_pages(self) -> Dict[str, List[Path]]:
        """
        Discover all pages and their patches in the folder
        
        Returns:
            Dictionary mapping page names to list of patch files
        """
        print("\n" + "="*80)
        print("DISCOVERING PAGES IN FOLDER")
        print("="*80)
        
        all_patches = list(self.patch_folder.glob("*_r*_c*.png"))
        print(f"Total patch files found: {len(all_patches)}")
        
        if not all_patches:
            print("⚠ No patch files found matching pattern '*_r*_c*.png'")
            return {}
        
        # Group patches by page name
        pages = {}
        for patch_file in all_patches:
            # Extract page name (everything before _r)
            if '_r' in patch_file.name:
                page_name = patch_file.name.split('_r')[0]
                if page_name not in pages:
                    pages[page_name] = []
                pages[page_name].append(patch_file)
        
        # Sort patches for each page
        for page_name in pages:
            pages[page_name] = sorted(pages[page_name])
        
        print(f"\nDiscovered {len(pages)} page(s):")
        for page_name in sorted(pages.keys()):
            print(f"  • {page_name}: {len(pages[page_name])} patches")
        
        print("="*80 + "\n")
        
        return pages

    def detect_grid_size(self, patch_files: List[Path]) -> Tuple[int, int]:
        """
        Detect grid size (rows × columns) from patch file names
        
        Returns:
            (max_rows, max_cols)
        """
        max_row = 0
        max_col = 0
        
        for patch_file in patch_files:
            row, col = self.get_patch_position(patch_file.name)
            if row is not None:
                max_row = max(max_row, row)
            if col is not None:
                max_col = max(max_col, col)
        
        # Add 1 because indices are 0-based
        return max_row + 1, max_col + 1

    def normalize_identifier(self, size, code, number, suffix=None):
        """Normalize and fix OCR errors"""
        code = code.upper()

        # Fix OCR errors
        if code == '0W': code = 'OW'
        elif code == '0C': code = 'OC'
        elif code == 'C0': code = 'CO'
        elif code == 'CQ': code = 'CG'
        elif code == 'NC': code = 'NG'

        if not (len(code) == 2 and code.isalpha()):
            return None

        number = number.replace('O', '0').replace('I', '1')
        if not (4 <= len(number) <= 6 and number.isdigit()):
            return None

        if suffix:
            suffix = suffix.upper().replace('O', '0').replace('I', '1')
            if not (2 <= len(suffix) <= 4 and suffix[0].isalpha() and suffix[1:].isdigit()):
                return None
            return f'{size}"-{code}-{number}-{suffix}'

        return None

    def extract_text_from_predictions(self, predictions) -> List[Tuple[str, tuple]]:
        """Extract text with bboxes from Surya predictions"""
        results = []
        for line in predictions[0].text_lines:
            text = line.text.strip()
            if text:
                # bbox format: (x_min, y_min, x_max, y_max)
                results.append((text, line.bbox))
        return results

    def extract_complete_identifiers(self, texts_with_bboxes: List[Tuple[str, tuple]]) -> Set[str]:
        """Extract complete ISO identifiers"""
        found = set()

        for text, _ in texts_with_bboxes:
            match = self.complete_pattern.search(text)
            if match:
                size = match.group(1)
                code = match.group(2)
                number = match.group(3)
                suffix = match.group(4)

                formatted = self.normalize_identifier(size, code, number, suffix)
                if formatted:
                    found.add(formatted)

        return found

    def has_partial_pattern(self, text: str) -> bool:
        """
        Strict check for partial ISO identifier patterns
        """
        text = text.strip()

        # Must contain dash
        if '-' not in text:
            return False

        # If contains quote mark ("), it's likely an ISO identifier with size
        if '"' in text:
            # Check against partial patterns
            for pattern in self.partial_patterns:
                if pattern.match(text):
                    return True
            return False

        # No quote mark - more careful checking needed

        # Reject equipment tags (2-3 letters + dash + exactly 4 digits, nothing else)
        equipment_tag_pattern = re.compile(r'^[A-Z]{2,3}-\d{4}$', re.I)
        if equipment_tag_pattern.match(text):
            return False

        # Reject document numbers (contains more than 4 dash-separated parts)
        parts = text.split('-')
        if len(parts) > 4:
            return False

        # Additional check: if it has exactly 4 parts and none look like ISO components, reject
        if len(parts) == 4:
            has_long_number = any(part.isdigit() and len(part) >= 4 for part in parts)
            if not has_long_number:
                return False

        # Check against valid partial patterns
        for pattern in self.partial_patterns:
            if pattern.match(text):
                return True

        return False

    def get_patch_position(self, patch_name: str) -> Tuple[Optional[int], Optional[int]]:
        """Extract row and column from patch name"""
        match = re.search(r'_r(\d+)_c(\d+)', patch_name)
        if match:
            return int(match.group(1)), int(match.group(2))
        return None, None

    def determine_edge_and_direction(self, bbox: tuple, width: int, height: int,
                                     edge_threshold: int = 80) -> Tuple[Optional[str], Optional[str]]:
        """
        Determine edge location and extension direction
        """
        x_min, y_min, x_max, y_max = bbox

        # Check which edge the bbox is closest to
        if y_min < edge_threshold:
            return 'top', 'up'
        elif y_max > (height - edge_threshold):
            return 'bottom', 'down'
        elif x_min < edge_threshold:
            return 'left', 'left'
        elif x_max > (width - edge_threshold):
            return 'right', 'right'

        return None, None

    def get_neighbor_patch(self, patch_name: str, direction: str, max_rows: int, max_cols: int) -> Optional[str]:
        """Get neighbor patch name based on direction and grid size"""
        row, col = self.get_patch_position(patch_name)
        if row is None:
            return None

        # Calculate neighbor position
        if direction == 'up':
            row -= 1
        elif direction == 'down':
            row += 1
        elif direction == 'left':
            col -= 1
        elif direction == 'right':
            col += 1

        # Check bounds (dynamic based on detected grid size)
        if row < 0 or row >= max_rows or col < 0 or col >= max_cols:
            return None

        base_name = patch_name.split('_r')[0]
        return f"{base_name}_r{row}_c{col}.png"

    def create_targeted_extension(self, current_patch: np.ndarray,
                                  neighbor_patch: np.ndarray,
                                  bbox: tuple,
                                  direction: str,
                                  padding: int = 40,
                                  lateral_margin: int = 50) -> Optional[np.ndarray]:
        """
        Create a targeted extended region around the partial identifier
        """
        # Convert bbox to integers
        x_min, y_min, x_max, y_max = map(int, bbox)
        h_curr, w_curr = current_patch.shape[:2]
        h_neigh, w_neigh = neighbor_patch.shape[:2]

        # Add lateral margins (convert to int)
        x_min_expanded = int(max(0, x_min - lateral_margin))
        x_max_expanded = int(min(w_curr, x_max + lateral_margin))
        y_min_expanded = int(max(0, y_min - lateral_margin))
        y_max_expanded = int(min(h_curr, y_max + lateral_margin))

        if direction == 'up':
            # Extend upward into neighbor's bottom region
            x_min_neigh = int(max(0, x_min_expanded))
            x_max_neigh = int(min(w_neigh, x_max_expanded))
            neighbor_strip = neighbor_patch[-padding:, x_min_neigh:x_max_neigh]

            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]
            extended = np.vstack([neighbor_strip, current_region])

        elif direction == 'down':
            # Extend downward into neighbor's top region
            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]

            x_min_neigh = int(max(0, x_min_expanded))
            x_max_neigh = int(min(w_neigh, x_max_expanded))
            neighbor_strip = neighbor_patch[:padding, x_min_neigh:x_max_neigh]

            extended = np.vstack([current_region, neighbor_strip])

        elif direction == 'left':
            # Extend leftward into neighbor's right region
            y_min_neigh = int(max(0, y_min_expanded))
            y_max_neigh = int(min(h_neigh, y_max_expanded))
            neighbor_strip = neighbor_patch[y_min_neigh:y_max_neigh, -padding:]

            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]
            extended = np.hstack([neighbor_strip, current_region])

        elif direction == 'right':
            # Extend rightward into neighbor's left region
            current_region = current_patch[y_min_expanded:y_max_expanded, x_min_expanded:x_max_expanded]

            y_min_neigh = int(max(0, y_min_expanded))
            y_max_neigh = int(min(h_neigh, y_max_expanded))
            neighbor_strip = neighbor_patch[y_min_neigh:y_max_neigh, :padding]

            extended = np.hstack([current_region, neighbor_strip])

        else:
            return None

        return extended

    def run_ocr_on_image(self, img: np.ndarray) -> Tuple[Set[str], List[Tuple[str, tuple]]]:
        """Run OCR on image array"""
        # Convert BGR to RGB for PIL
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)

        # Run OCR
        predictions = self.rec_predictor([img_pil], det_predictor=self.det_predictor)

        # Extract text with bboxes
        texts_with_bboxes = self.extract_text_from_predictions(predictions)

        # Find complete identifiers
        complete_ids = self.extract_complete_identifiers(texts_with_bboxes)

        return complete_ids, texts_with_bboxes

    def process_single_patch(self, patch_path: Path) -> Tuple[Set[str], List[dict]]:
        """Process a single patch to find complete and partial identifiers"""
        print(f"\nProcessing: {patch_path.name}")

        # Load patch
        patch = cv2.imread(str(patch_path))
        if patch is None:
            print(f"  ERROR: Could not load patch")
            return set(), []

        height, width = patch.shape[:2]

        # Run OCR
        complete_ids, texts_with_bboxes = self.run_ocr_on_image(patch)

        if complete_ids:
            print(f"  ✓ Complete: {complete_ids}")

        # Debug: Show all detected text
        print(f"  📝 All detected text ({len(texts_with_bboxes)} items):")
        for text, bbox in texts_with_bboxes[:10]:  # Show first 10 only
            print(f"     '{text}'")
        if len(texts_with_bboxes) > 10:
            print(f"     ... and {len(texts_with_bboxes) - 10} more")

        # Find partials
        partials_info = []
        for text, bbox in texts_with_bboxes:
            # Skip if part of complete identifier
            if any(text in cid for cid in complete_ids):
                continue

            # Check if partial pattern
            if self.has_partial_pattern(text):
                edge, direction = self.determine_edge_and_direction(bbox, width, height)

                if edge and direction:
                    print(f"  ⚠ Partial on {edge}: '{text}' at bbox {bbox}")
                    partials_info.append({
                        'text': text,
                        'bbox': bbox,
                        'edge': edge,
                        'direction': direction
                    })
                else:
                    # Partial found but not on edge
                    print(f"  ℹ Partial (not on edge): '{text}'")

        return complete_ids, partials_info

    def process_extensions(self, patch_path: Path, partials_info: List[dict], 
                          max_rows: int, max_cols: int) -> Set[str]:
        """Process targeted extensions for partial identifiers"""
        found_in_extensions = set()

        if not partials_info:
            return found_in_extensions

        # Load current patch once
        current_patch = cv2.imread(str(patch_path))
        if current_patch is None:
            return found_in_extensions

        for partial in partials_info:
            text = partial['text']
            bbox = partial['bbox']
            direction = partial['direction']

            # Get neighbor patch
            neighbor_name = self.get_neighbor_patch(patch_path.name, direction, max_rows, max_cols)
            if not neighbor_name:
                print(f"  ⚠ No neighbor for '{text}' in {direction} direction")
                continue

            neighbor_path = self.patch_folder / neighbor_name
            if not neighbor_path.exists():
                print(f"  ⚠ Neighbor not found: {neighbor_name}")
                continue

            # Load neighbor patch
            neighbor_patch = cv2.imread(str(neighbor_path))
            if neighbor_patch is None:
                print(f"  ⚠ Could not load neighbor: {neighbor_name}")
                continue

            print(f"  → Creating targeted extension to {neighbor_name} ({direction}) for '{text}'")

            # Create targeted extended region
            extended_region = self.create_targeted_extension(
                current_patch,
                neighbor_patch,
                bbox,
                direction,
                padding=40,
                lateral_margin=50
            )

            if extended_region is None or extended_region.size == 0:
                print(f"  ✗ Failed to create extended region")
                continue

            # Run OCR on small extended region
            try:
                complete_ids, _ = self.run_ocr_on_image(extended_region)

                if complete_ids:
                    print(f"  ✓ Found in extension: {complete_ids}")
                    found_in_extensions.update(complete_ids)
                else:
                    print(f"  ⚠ No complete identifiers in extension")

            except Exception as e:
                print(f"  ✗ OCR error: {e}")

        return found_in_extensions

    def extract_from_page(self, page_name: str, patch_files: List[Path]) -> dict:
        """Extract all ISO identifiers from a single page"""
        print("="*80)
        print(f"EXTRACTING FROM PAGE: {page_name}")
        print("="*80)

        print(f"\nTotal patches: {len(patch_files)}")
        
        # Detect grid size
        max_rows, max_cols = self.detect_grid_size(patch_files)
        print(f"Detected grid size: {max_rows} rows × {max_cols} columns")

        all_complete = set()
        all_from_extensions = set()

        # Process each patch
        for patch_path in patch_files:
            # Get complete and partials from patch
            complete_ids, partials_info = self.process_single_patch(patch_path)
            all_complete.update(complete_ids)

            # Process extensions for partials
            if partials_info:
                extension_ids = self.process_extensions(patch_path, partials_info, max_rows, max_cols)
                all_from_extensions.update(extension_ids)

        # Combine results
        all_identifiers = all_complete | all_from_extensions

        return {
            'page_name': page_name,
            'total_patches': len(patch_files),
            'grid_size': f"{max_rows}×{max_cols}",
            'identifiers': sorted(all_identifiers),
            'total_found': len(all_identifiers),
            'from_complete': len(all_complete),
            'from_extensions': len(all_from_extensions),
        }

    def extract_from_all_pages(self) -> Dict[str, dict]:
        """
        Extract ISO identifiers from all pages in the folder
        
        Returns:
            Dictionary mapping page names to their results
        """
        # Discover all pages
        pages = self.discover_pages()
        
        if not pages:
            print("⚠ No pages found!")
            return {}
        
        all_results = {}
        
        # Process each page
        for page_name in sorted(pages.keys()):
            patch_files = pages[page_name]
            results = self.extract_from_page(page_name, patch_files)
            all_results[page_name] = results
        
        return all_results


def check_cuda():
    """Verify CUDA availability and GPU info"""
    print("\n" + "="*80)
    print("CUDA CHECK")
    print("="*80)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("⚠ WARNING: CUDA not available - will use CPU")
        print("  Install CUDA-enabled PyTorch:")
        print("  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118")
    print("="*80 + "\n")


def main():
    # Configuration
    patch_folder = r"E:\Projects\P&ID Versions\P&ID V2\iso-identifier-test-patches\test-node1.4-patches"

    # Create extractor
    extractor = TargetedPatchExtractor(patch_folder)

    # Extract from all pages
    all_results = extractor.extract_from_all_pages()

    # Print combined summary
    print("\n" + "="*80)
    print("FINAL RESULTS - ALL PAGES")
    print("="*80)
    
    total_pages = len(all_results)
    total_patches = sum(r['total_patches'] for r in all_results.values())
    total_identifiers = sum(r['total_found'] for r in all_results.values())
    total_from_complete = sum(r['from_complete'] for r in all_results.values())
    total_from_extensions = sum(r['from_extensions'] for r in all_results.values())
    
    print(f"\nTotal pages processed: {total_pages}")
    print(f"Total patches processed: {total_patches}")
    print(f"Total identifiers found: {total_identifiers}")
    print(f"  - From complete patches: {total_from_complete}")
    print(f"  - From targeted extensions: {total_from_extensions}")
    
    # Print per-page results
    print("\n" + "-"*80)
    print("PER-PAGE BREAKDOWN")
    print("-"*80)
    
    for page_name in sorted(all_results.keys()):
        results = all_results[page_name]
        print(f"\n📄 {page_name} ({results['grid_size']}):")
        print(f"   Patches: {results['total_patches']}")
        print(f"   Identifiers: {results['total_found']} (Complete: {results['from_complete']}, Extensions: {results['from_extensions']})")
        
        if results['identifiers']:
            print(f"   Found:")
            for idx, identifier in enumerate(results['identifiers'], 1):
                print(f"     {idx:2d}. {identifier}")
        else:
            print(f"   ⚠ No identifiers found")
    
    # Check critical identifiers across all pages
    print("\n" + "="*80)
    print("CHECKING CRITICAL IDENTIFIERS")
    print("="*80)

    critical = [
        "1/2\"-CP-8113-D48",
        "6\"-OW-8104-D03",
        "8\"-OW-8105-D03",
        "6\"-OW-8601-D03",
        "8\"-CL-8903-D03",
        "24\"-CG-8111",
    ]

    # Combine all identifiers from all pages
    all_identifiers = set()
    for results in all_results.values():
        all_identifiers.update(results['identifiers'])

    for c in critical:
        if c in all_identifiers:
            # Find which page(s) it's on
            pages_found = []
            for page_name, results in all_results.items():
                if c in results['identifiers']:
                    pages_found.append(page_name)
            print(f"  ✓ {c} - FOUND (on {', '.join(pages_found)})")
        else:
            # Check for similar (size might be wrong)
            similar = [v for v in all_identifiers if c.split('-')[1:] == v.split('-')[1:] if '-' in v and '-' in c]
            if similar:
                print(f"  ~ {c} - Similar: {similar[0]}")
            else:
                print(f"  ✗ {c} - MISSING")

    print("\n" + "="*80)

    return all_results


if __name__ == "__main__":
    check_cuda()
    main()


CUDA CHECK
PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU Device: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Memory: 4.29 GB


✓ Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
  CUDA Version: 11.8
  GPU Memory: 4.29 GB

Initializing OCR...


RuntimeError: CUDA error: an illegal memory access was encountered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [3]:
#iso-legends-extractor-v4
"""
ISO Legend Extractor - Extract ISO definitions and their colors from legend images
"""

import numpy as np
import cv2
import re
from collections import defaultdict
from surya.foundation import FoundationPredictor
from surya.detection import DetectionPredictor
from surya.recognition import RecognitionPredictor
from PIL import Image
import colorsys


# ----------------------------
# Color Utility Functions
# ----------------------------

def rgb_to_hex(rgb):
    """Convert RGB tuple to hex string"""
    return '#%02x%02x%02x' % tuple(int(x) for x in rgb)


def rgb_to_hsv(rgb):
    """Convert RGB to HSV"""
    r, g, b = [x / 255.0 for x in rgb]
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    return (int(h * 180), int(s * 255), int(v * 255))


def is_colorful(rgb, brightness_thresh=30, color_diff_thresh=20):
    """Check if a color is colorful (not grayscale)"""
    r, g, b = rgb
    brightness = np.mean(rgb)
    color_diff = np.max(rgb) - np.min(rgb)
    return (brightness > brightness_thresh) and (brightness < 240) and (color_diff > color_diff_thresh)


def detect_line_pattern(color_region, iso_name="unknown"):
    """
    Detect if a color region represents a solid or dashed line.
    Returns: 'solid', 'dashed', or 'unknown'
    """
    if color_region.size == 0:
        return 'unknown'

    # Get the height and width
    h, w = color_region.shape[:2]

    if h < 2 or w < 10:  # Too small to analyze
        return 'unknown'

    # Sample multiple horizontal lines (not just middle) for better accuracy
    sample_rows = []
    if h >= 6:
        sample_rows = [h // 4, h // 2, 3 * h // 4]
    elif h >= 3:
        sample_rows = [h // 3, 2 * h // 3]
    else:
        sample_rows = [h // 2]

    all_gaps = []
    all_coverage = []

    for row_idx in sample_rows:
        if row_idx >= h:
            continue

        # Get the row
        row = color_region[row_idx, :]

        # Build a mask of colorful pixels
        colorful_mask = np.array([is_colorful(pixel) for pixel in row])

        colored_count = np.sum(colorful_mask)

        if colored_count < 2:  # Not enough colored pixels
            continue

        # Find the first and last colorful pixel to focus on the actual line
        colored_indices = np.where(colorful_mask)[0]
        if len(colored_indices) < 2:
            continue

        start_idx = colored_indices[0]
        end_idx = colored_indices[-1]

        # Analyze only the region between first and last colored pixel
        line_segment = colorful_mask[start_idx:end_idx+1]

        if len(line_segment) < 5:  # Line too short
            continue

        # Count gaps (consecutive False values surrounded by True values)
        gaps = 0
        in_gap = False
        gap_lengths = []
        current_gap_length = 0

        for val in line_segment:
            if not val:  # In a gap
                if not in_gap:
                    gaps += 1
                    in_gap = True
                    current_gap_length = 1
                else:
                    current_gap_length += 1
            else:  # Colored pixel
                if in_gap:
                    gap_lengths.append(current_gap_length)
                    current_gap_length = 0
                in_gap = False

        # Calculate coverage in the line segment
        coverage = np.sum(line_segment) / len(line_segment)

        all_gaps.append(gaps)
        all_coverage.append(coverage)

    if not all_gaps:
        # Fallback: analyze the entire region as a whole
        total_colorful = 0
        for row in color_region:
            for pixel in row:
                if is_colorful(pixel):
                    total_colorful += 1

        total_pixels = h * w
        overall_coverage = total_colorful / total_pixels if total_pixels > 0 else 0

        # More aggressive fallback thresholds
        if overall_coverage > 0.3:
            return 'solid'
        elif overall_coverage > 0.1:
            return 'dashed'
        else:
            return 'solid'  # Default to solid if very low coverage

    # Average across all sampled rows
    avg_gaps = np.mean(all_gaps)
    avg_coverage = np.mean(all_coverage)

    # More lenient classification logic:
    # Dashed lines: have clear gaps with reasonable coverage
    # Solid lines: high coverage with few/no gaps

    if avg_gaps >= 3:
        # Multiple gaps = definitely dashed
        return 'dashed'
    elif avg_gaps >= 1.5 and avg_coverage < 0.85:
        # Some gaps with not-full coverage = dashed
        return 'dashed'
    elif avg_gaps < 1 and avg_coverage > 0.8:
        # Very few gaps and high coverage = solid
        return 'solid'
    elif avg_coverage >= 0.85:
        # Very high coverage = solid (even with a small gap)
        return 'solid'
    elif avg_coverage < 0.6:
        # Low coverage = dashed
        return 'dashed'
    else:
        # Middle ground: use gaps as primary indicator
        if avg_gaps >= 1:
            return 'dashed'
        else:
            return 'solid'


# ----------------------------
# ISO Legend Detection Function
# ----------------------------

def detect_iso_texts_and_colors(image_path, debug=False, save_color_regions=False):
    """
    Detect ISO line texts and their associated colors from legend page using Surya OCR.
    Returns dictionary mapping ISO names to their RGB/HSV colors.

    Args:
        image_path: Path to the legend image
        debug: If True, saves debug visualization

    Returns:
        iso_color_map: Dictionary with structure:
            {
                'CP ISO 14.1': {
                    'rgb': (255, 255, 0),
                    'hsv': (30, 255, 255),
                    'hex': '#ffff00'
                },
                ...
            }
    """
    # Initialize Surya predictors
    foundation = FoundationPredictor()
    det_predictor = DetectionPredictor()
    rec_predictor = RecognitionPredictor(foundation)

    # Load image
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    # Convert to PIL for Surya OCR
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(img_rgb)

    # Run Surya OCR
    predictions = rec_predictor([pil_image], det_predictor=det_predictor)
    text_lines = predictions[0].text_lines

    # Simplified output
    print(f"Detected {len(text_lines)} text lines from legend image")

    # Extract ISO text boxes - SUPPORT MULTIPLE PATTERNS
    iso_boxes = []

    # Define multiple node identifier patterns
    node_patterns = [
        r'GBN-CPP-\d+',              # GBN-CPP-01, GBN-CPP-02, etc.
        r'CP\s*ISO\s*[\d.]+',        # CP ISO 14.1, CP ISO 16.1, etc.
        r'BDD-CPP-\d+',              # BDD-CPP-01, BDD-CPP-02, etc.
        r'\d+\.',                    # 1., 2., 3., etc.
        r'\d+$',                     # 1, 2, 3, etc. (standalone numbers)
    ]

    # Match patterns silently
    matched_count = 0
    for line in text_lines:
        text = line.text.strip()

        # Try each pattern
        iso_name = None
        matched_pattern = None
        for pattern in node_patterns:
            match = re.search(pattern, text)
            if match:
                iso_name = match.group(0)
                matched_pattern = pattern
                break  # Found a match, stop trying patterns

        if iso_name:
            # Special filtering for standalone number pattern (\d+$)
            if matched_pattern == r'\d+$':
                # Only accept if the ENTIRE text is just the number (or very short like "1", "2", etc.)
                # This filters out numbers that are parts of longer strings
                if len(text) > 5:  # If text is longer than 5 chars, it's probably not a standalone node number
                    continue
                # Additional check: ensure the matched number IS the entire text (or very close)
                if text != iso_name and not re.match(r'^\d+\.?$', text):
                    continue

            # Surya bbox is in pixel coordinates: [x1, y1, x2, y2]
            bbox = line.bbox
            x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
            iso_boxes.append((iso_name, (x1, y1, x2, y2)))
            matched_count += 1

    print(f"Matched {matched_count} ISO identifiers\n")

    # Extract color samples from regions adjacent to text
    iso_color_map = {}

    if debug:
        debug_img = img.copy()

    for iso_name, (x1, y1, x2, y2) in iso_boxes:
        # Strategy: Search across the entire row to the right of the text
        # This handles various legend layouts where colors might be in different columns

        # Expand search region - look across most of the row width
        color_region_y1 = y1 - 2  # Slight vertical tolerance
        color_region_y2 = y2 + 2

        # Try multiple horizontal search regions, starting from the text and extending rightward
        search_regions = [
            (x2 + 5, min(x2 + 150, w)),      # Close to text (original approach)
            (x2 + 100, min(x2 + 400, w)),    # Medium distance
            (int(w * 0.5), w - 10),          # Right half of image
            (int(w * 0.6), w - 10),          # Right 40% of image
        ]

        best_color = None
        best_colorful_count = 0
        best_region_coords = None
        best_pattern = 'unknown'
        best_color_region = None

        for region_x1, region_x2 in search_regions:
            if region_x1 >= region_x2:
                continue

            # Extract color region
            color_region = img_rgb[color_region_y1:color_region_y2, region_x1:region_x2]

            if color_region.size == 0:
                continue

            # Get dominant color from this region
            pixels = color_region.reshape(-1, 3)
            colorful_pixels = np.array([p for p in pixels if is_colorful(p)])

            # Keep track of the region with the most colorful pixels
            if len(colorful_pixels) > best_colorful_count:
                best_colorful_count = len(colorful_pixels)
                best_color = np.median(colorful_pixels, axis=0).astype(int)
                best_region_coords = (region_x1, color_region_y1, region_x2, color_region_y2)
                best_color_region = color_region

        # Use the best color found
        if best_color is not None and best_colorful_count > 10:  # Minimum pixel threshold
            avg_color_tuple = tuple(best_color)
            hsv_color = rgb_to_hsv(avg_color_tuple)

            # Detect line pattern (solid/dashed)
            pattern = detect_line_pattern(best_color_region, iso_name) if best_color_region is not None else 'unknown'

            # Save color region for debugging if requested
            if save_color_regions and best_color_region is not None:
                import os
                os.makedirs('color_regions', exist_ok=True)
                region_path = f'color_regions/{iso_name.replace("/", "_")}.png'
                cv2.imwrite(region_path, cv2.cvtColor(best_color_region, cv2.COLOR_RGB2BGR))

            # Normalize ISO name
            iso_normalized = re.sub(r'\s+', ' ', iso_name.upper())
            iso_color_map[iso_normalized] = {
                'rgb': avg_color_tuple,
                'hsv': hsv_color,
                'hex': rgb_to_hex(avg_color_tuple),
                'pattern': pattern
            }

            if debug and best_region_coords:
                cv2.rectangle(debug_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                region_x1, region_y1, region_x2, region_y2 = best_region_coords
                cv2.rectangle(debug_img, (region_x1, region_y1),
                            (region_x2, region_y2), (255, 0, 0), 2)
                cv2.putText(debug_img, iso_normalized, (x1, y1-5),
                          cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    if debug and iso_boxes:
        cv2.imwrite('legend_debug.png', debug_img)

    return iso_color_map


# ----------------------------
# Example Usage
# ----------------------------

if __name__ == "__main__":
    # Example: Extract ISO definitions from legend
    legend_path = input("Enter the path to your ISO legend image: ").strip()

    print(f"\nExtracting ISO definitions from: {legend_path}")
    iso_color_map = detect_iso_texts_and_colors(legend_path, debug=True, save_color_regions=True)

    if not iso_color_map:
        print("\n⚠ No ISO definitions found in legend.")
    else:
        print(f"\n{'='*70}")
        print(f"{'ISO Definition':<20} {'Color':<10} {'Pattern':<10}")
        print(f"{'='*70}")

        for iso_name, color_info in sorted(iso_color_map.items()):
            pattern = color_info.get('pattern', 'unknown')
            print(f"{iso_name:<20} {color_info['hex']:<10} {pattern:<10}")

        print(f"{'='*70}")
        print(f"Total: {len(iso_color_map)} ISO definitions extracted\n")


Extracting ISO definitions from: C:\Users\saroc\Desktop\P&ID V2\legends-sheets\GBN-EPIC-ERCP_1.png


Recognizing Text: 100%|██████████| 65/65 [00:05<00:00, 10.90it/s]


Detected 65 text lines from legend image
Matched 17 ISO identifiers


ISO Definition       Color      Pattern   
GBN-CPP-01           #f69786    solid     
GBN-CPP-02           #fed47e    solid     
GBN-CPP-03           #f59385    dashed    
GBN-CPP-04           #d3df7d    solid     
GBN-CPP-05           #88d6f6    solid     
GBN-CPP-06           #d3df7d    dashed    
GBN-CPP-07           #88d6f6    dashed    
GBN-CPP-08           #fed47e    solid     
GBN-CPP-09           #f7f17e    solid     
GBN-CPP-10           #bc7eb6    solid     
GBN-CPP-11           #dcd2e8    solid     
GBN-CPP-12           #f7bed7    solid     
GBN-CPP-13           #fcded3    solid     
GBN-CPP-14           #f7bed7    dashed    
GBN-CPP-15           #fed47e    dashed    
GBN-CPP-16           #bc7fb6    dashed    
GBN-CPP-22           #98d8e5    dashed    
Total: 17 ISO definitions extracted



In [5]:
#iso-identifier-v4-color-optimized
"""
Color-Optimized ISO Identifier Extraction
Skips patches without colored lines and extracts only identifiers adjacent to colored ISO lines
Associates identifiers with their legend entries based on line color
"""

import cv2
import numpy as np
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.detection import DetectionPredictor
from surya.recognition import RecognitionPredictor
import re
from pathlib import Path
from typing import List, Set, Tuple, Optional, Dict
import torch
import os
import colorsys

# Fix for CUDNN_STATUS_BAD_PARAM_STREAM_MISMATCH
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


# ============================================================================
# COLOR UTILITY FUNCTIONS (from iso-legends-extractor-v4)
# ============================================================================

def rgb_to_hex(rgb):
    """Convert RGB tuple to hex string"""
    return '#%02x%02x%02x' % tuple(int(x) for x in rgb)


def rgb_to_hsv(rgb):
    """Convert RGB to HSV"""
    r, g, b = [x / 255.0 for x in rgb]
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    return (int(h * 180), int(s * 255), int(v * 255))


def is_colorful(rgb, brightness_thresh=30, color_diff_thresh=20):
    """Check if a color is colorful (not grayscale)"""
    r, g, b = rgb
    brightness = np.mean(rgb)
    color_diff = np.max(rgb) - np.min(rgb)
    return (brightness > brightness_thresh) and (brightness < 240) and (color_diff > color_diff_thresh)


def color_distance(rgb1, rgb2):
    """Calculate Euclidean distance between two RGB colors"""
    return np.sqrt(np.sum((np.array(rgb1) - np.array(rgb2)) ** 2))


def hsv_distance(hsv1, hsv2):
    """Calculate distance between two HSV colors (weighted for hue circularity)"""
    h1, s1, v1 = hsv1
    h2, s2, v2 = hsv2
    
    # Hue distance (circular)
    dh = min(abs(h1 - h2), 180 - abs(h1 - h2)) / 180.0
    ds = abs(s1 - s2) / 255.0
    dv = abs(v1 - v2) / 255.0
    
    # Weighted distance (hue most important for line colors)
    return np.sqrt((dh * 2.0) ** 2 + ds ** 2 + dv ** 2)


def detect_line_pattern(color_region, iso_name="unknown"):
    """
    Detect if a color region represents a solid or dashed line.
    Returns: 'solid', 'dashed', or 'unknown'
    """
    if color_region.size == 0:
        return 'unknown'

    h, w = color_region.shape[:2]

    if h < 2 or w < 10:
        return 'unknown'

    # Sample multiple horizontal lines
    sample_rows = []
    if h >= 6:
        sample_rows = [h // 4, h // 2, 3 * h // 4]
    elif h >= 3:
        sample_rows = [h // 3, 2 * h // 3]
    else:
        sample_rows = [h // 2]

    all_gaps = []
    all_coverage = []

    for row_idx in sample_rows:
        if row_idx >= h:
            continue

        row = color_region[row_idx, :]
        colorful_mask = np.array([is_colorful(pixel) for pixel in row])
        colored_count = np.sum(colorful_mask)

        if colored_count < 2:
            continue

        colored_indices = np.where(colorful_mask)[0]
        if len(colored_indices) < 2:
            continue

        start_idx = colored_indices[0]
        end_idx = colored_indices[-1]
        line_segment = colorful_mask[start_idx:end_idx+1]

        if len(line_segment) < 5:
            continue

        gaps = 0
        in_gap = False
        gap_lengths = []
        current_gap_length = 0

        for val in line_segment:
            if not val:
                if not in_gap:
                    gaps += 1
                    in_gap = True
                    current_gap_length = 1
                else:
                    current_gap_length += 1
            else:
                if in_gap:
                    gap_lengths.append(current_gap_length)
                    current_gap_length = 0
                in_gap = False

        coverage = np.sum(line_segment) / len(line_segment)
        all_gaps.append(gaps)
        all_coverage.append(coverage)

    if not all_gaps:
        total_colorful = 0
        for row in color_region:
            for pixel in row:
                if is_colorful(pixel):
                    total_colorful += 1

        total_pixels = h * w
        overall_coverage = total_colorful / total_pixels if total_pixels > 0 else 0

        if overall_coverage > 0.3:
            return 'solid'
        elif overall_coverage > 0.1:
            return 'dashed'
        else:
            return 'solid'

    avg_gaps = np.mean(all_gaps)
    avg_coverage = np.mean(all_coverage)

    if avg_gaps >= 3:
        return 'dashed'
    elif avg_gaps >= 1.5 and avg_coverage < 0.85:
        return 'dashed'
    elif avg_gaps < 1 and avg_coverage > 0.8:
        return 'solid'
    elif avg_coverage >= 0.85:
        return 'solid'
    elif avg_coverage < 0.6:
        return 'dashed'
    else:
        if avg_gaps >= 1:
            return 'dashed'
        else:
            return 'solid'


# ============================================================================
# LEGEND EXTRACTION (from iso-legends-extractor-v4)
# ============================================================================

def detect_iso_texts_and_colors(image_path, debug=False):
    """
    Extract ISO legend entries and their colors
    Returns dictionary mapping ISO legend names to color info
    """
    foundation = FoundationPredictor()
    det_predictor = DetectionPredictor()
    rec_predictor = RecognitionPredictor(foundation)

    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(img_rgb)

    predictions = rec_predictor([pil_image], det_predictor=det_predictor)
    text_lines = predictions[0].text_lines

    print(f"  Detected {len(text_lines)} text lines from legend")

    # Extract node identifiers
    iso_boxes = []
    node_patterns = [
        r'GBN-CPP-\d+',
        r'BDD-CPP-\d+',
        r'CP\s*ISO\s*[\d.]+',
        r'\d+\.',
        r'\d+$',
    ]

    for line in text_lines:
        text = line.text.strip()
        iso_name = None
        matched_pattern = None

        for pattern in node_patterns:
            match = re.search(pattern, text)
            if match:
                iso_name = match.group(0)
                matched_pattern = pattern
                break

        if iso_name:
            if matched_pattern == r'\d+$':
                if len(text) > 5:
                    continue
                if text != iso_name and not re.match(r'^\d+\.?$', text):
                    continue

            bbox = line.bbox
            x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
            iso_boxes.append((iso_name, (x1, y1, x2, y2)))

    # Extract colors
    iso_color_map = {}

    if debug:
        debug_img = img.copy()

    for iso_name, (x1, y1, x2, y2) in iso_boxes:
        color_region_y1 = y1 - 2
        color_region_y2 = y2 + 2

        search_regions = [
            (x2 + 5, min(x2 + 150, w)),
            (x2 + 100, min(x2 + 400, w)),
            (int(w * 0.5), w - 10),
            (int(w * 0.6), w - 10),
        ]

        best_color = None
        best_colorful_count = 0
        best_region_coords = None
        best_color_region = None

        for region_x1, region_x2 in search_regions:
            if region_x1 >= region_x2:
                continue

            color_region = img_rgb[color_region_y1:color_region_y2, region_x1:region_x2]

            if color_region.size == 0:
                continue

            pixels = color_region.reshape(-1, 3)
            colorful_pixels = np.array([p for p in pixels if is_colorful(p)])

            if len(colorful_pixels) > best_colorful_count:
                best_colorful_count = len(colorful_pixels)
                best_color = np.median(colorful_pixels, axis=0).astype(int)
                best_region_coords = (region_x1, color_region_y1, region_x2, color_region_y2)
                best_color_region = color_region

        if best_color is not None and best_colorful_count > 10:
            avg_color_tuple = tuple(best_color)
            hsv_color = rgb_to_hsv(avg_color_tuple)
            pattern = detect_line_pattern(best_color_region, iso_name) if best_color_region is not None else 'unknown'

            iso_normalized = re.sub(r'\s+', ' ', iso_name.upper())
            iso_color_map[iso_normalized] = {
                'rgb': avg_color_tuple,
                'hsv': hsv_color,
                'hex': rgb_to_hex(avg_color_tuple),
                'pattern': pattern
            }

            if debug and best_region_coords:
                cv2.rectangle(debug_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                region_x1, region_y1, region_x2, region_y2 = best_region_coords
                cv2.rectangle(debug_img, (region_x1, region_y1),
                            (region_x2, region_y2), (255, 0, 0), 2)

    if debug and iso_boxes:
        cv2.imwrite('legend_debug.png', debug_img)

    return iso_color_map


# ============================================================================
# COLORED LINE DETECTION IN PATCHES
# ============================================================================

def has_colored_lines(patch_image: np.ndarray, min_line_pixels: int = 100) -> bool:
    """
    Quick check if patch contains any colored lines
    Returns True if colored pixels exist above threshold
    """
    h, w = patch_image.shape[:2]
    
    # Convert to RGB
    if len(patch_image.shape) == 2:
        return False
    
    if patch_image.shape[2] == 4:  # RGBA
        patch_rgb = cv2.cvtColor(patch_image, cv2.COLOR_BGRA2RGB)
    else:
        patch_rgb = cv2.cvtColor(patch_image, cv2.COLOR_BGR2RGB)
    
    # Count colorful pixels
    colorful_count = 0
    
    # Sample every 4th pixel for speed
    for y in range(0, h, 4):
        for x in range(0, w, 4):
            pixel = patch_rgb[y, x]
            if is_colorful(pixel):
                colorful_count += 1
                
                # Early exit if threshold met
                if colorful_count * 16 > min_line_pixels:  # *16 because we sample every 4th
                    return True
    
    return colorful_count * 16 > min_line_pixels


def extract_colored_line_regions(patch_image: np.ndarray, 
                                  legend_colors: Dict,
                                  color_tolerance: float = 50.0) -> List[Dict]:
    """
    Extract regions containing colored lines that match legend colors
    
    Returns:
        List of dicts with:
        - 'bbox': (x1, y1, x2, y2)
        - 'color_rgb': detected color
        - 'legend_match': matched legend entry name
        - 'pattern': 'solid' or 'dashed'
    """
    h, w = patch_image.shape[:2]
    
    # Convert to RGB
    if patch_image.shape[2] == 4:
        patch_rgb = cv2.cvtColor(patch_image, cv2.COLOR_BGRA2RGB)
    else:
        patch_rgb = cv2.cvtColor(patch_image, cv2.COLOR_BGR2RGB)
    
    # Create mask of colorful pixels
    colorful_mask = np.zeros((h, w), dtype=np.uint8)
    
    for y in range(h):
        for x in range(w):
            if is_colorful(patch_rgb[y, x]):
                colorful_mask[y, x] = 255
    
    # Find connected components (line segments)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(colorful_mask, connectivity=8)
    
    line_regions = []
    
    for i in range(1, num_labels):  # Skip background (0)
        area = stats[i, cv2.CC_STAT_AREA]
        
        # Filter small regions
        if area < 50:
            continue
        
        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w_region = stats[i, cv2.CC_STAT_WIDTH]
        h_region = stats[i, cv2.CC_STAT_HEIGHT]
        
        # Extract region
        region_mask = (labels[y:y+h_region, x:x+w_region] == i).astype(np.uint8) * 255
        region_img = patch_rgb[y:y+h_region, x:x+w_region]
        
        # Get average color of this region
        masked_pixels = region_img[region_mask > 0]
        
        if len(masked_pixels) < 10:
            continue
        
        avg_color = np.median(masked_pixels, axis=0).astype(int)
        avg_color_tuple = tuple(avg_color)
        avg_hsv = rgb_to_hsv(avg_color_tuple)
        
        # Match to legend color
        best_match = None
        best_distance = float('inf')
        
        for legend_name, legend_info in legend_colors.items():
            legend_rgb = legend_info['rgb']
            legend_hsv = legend_info['hsv']
            
            # Use HSV distance for better color matching
            dist = hsv_distance(avg_hsv, legend_hsv)
            
            if dist < best_distance and dist < color_tolerance:
                best_distance = dist
                best_match = legend_name
        
        if best_match:
            # Detect pattern
            pattern = detect_line_pattern(region_img)
            
            line_regions.append({
                'bbox': (x, y, x + w_region, y + h_region),
                'color_rgb': avg_color_tuple,
                'legend_match': best_match,
                'pattern': pattern,
                'color_distance': best_distance
            })
    
    return line_regions


def is_text_near_line(text_bbox: Tuple, line_bbox: Tuple, proximity_threshold: int = 30) -> bool:
    """
    Check if text bounding box is near a colored line bounding box
    
    Args:
        text_bbox: (x1, y1, x2, y2) of text
        line_bbox: (x1, y1, x2, y2) of colored line
        proximity_threshold: maximum pixel distance
    
    Returns:
        True if text is adjacent to line
    """
    tx1, ty1, tx2, ty2 = text_bbox
    lx1, ly1, lx2, ly2 = line_bbox
    
    # Calculate text center
    text_center_x = (tx1 + tx2) / 2
    text_center_y = (ty1 + ty2) / 2
    
    # Check if text overlaps or is very close to line
    
    # Horizontal proximity (text left/right of line)
    horizontal_gap = min(
        abs(tx1 - lx2),  # Text to right of line
        abs(lx1 - tx2)   # Text to left of line
    )
    
    # Vertical proximity (text above/below line)
    vertical_gap = min(
        abs(ty1 - ly2),  # Text below line
        abs(ly1 - ty2)   # Text above line
    )
    
    # Vertical overlap check
    vertical_overlap = not (ty2 < ly1 or ty1 > ly2)
    
    # Horizontal overlap check
    horizontal_overlap = not (tx2 < lx1 or tx1 > lx2)
    
    # Text is "near" if:
    # 1. Vertically overlaps AND horizontally close
    # 2. OR horizontally overlaps AND vertically close
    
    if vertical_overlap and horizontal_gap < proximity_threshold:
        return True
    
    if horizontal_overlap and vertical_gap < proximity_threshold:
        return True
    
    # Also check if bboxes actually overlap
    if vertical_overlap and horizontal_overlap:
        return True
    
    return False


# ============================================================================
# OPTIMIZED PATCH EXTRACTOR
# ============================================================================

class ColorOptimizedPatchExtractor:
    """
    Optimized ISO identifier extraction with color-based filtering
    """

    def __init__(self, patch_folder: str, legend_path: str, 
                 patch_width: int = 640, patch_height: int = 640):
        self.patch_folder = Path(patch_folder)
        self.legend_path = legend_path
        self.patch_width = patch_width
        self.patch_height = patch_height

        # GPU setup
        os.environ['CUDA_VISIBLE_DEVICES'] = '0'
        
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            print(f"\n{'='*60}")
            print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
            print(f"{'='*60}\n")
        else:
            self.device = torch.device("cpu")
            print("\n⚠ CUDA not available, using CPU\n")

        print("Initializing OCR...")
        try:
            self.foundation = FoundationPredictor(device=self.device)
            self.det_predictor = DetectionPredictor(device=self.device)
            self.rec_predictor = RecognitionPredictor(self.foundation, device=self.device)
        except TypeError:
            self.foundation = FoundationPredictor()
            self.det_predictor = DetectionPredictor()
            self.rec_predictor = RecognitionPredictor(self.foundation)

        # ISO identifier patterns
        self.complete_pattern = re.compile(
            r'(\d+(?:/\d+)?)\"\s*-\s*([A-Z0-9]{2})\s*-\s*(\d{4,6})(?:\s*-\s*([A-Z]\d{1,3}))?',
            re.IGNORECASE
        )

        # Load legend colors
        print(f"\nLoading legend from: {legend_path}")
        self.legend_colors = detect_iso_texts_and_colors(legend_path, debug=True)
        
        if self.legend_colors:
            print(f"\n✓ Loaded {len(self.legend_colors)} legend entries:")
            for legend_name, color_info in sorted(self.legend_colors.items()):
                print(f"  • {legend_name:<20} {color_info['hex']:<10} ({color_info['pattern']})")
        else:
            print("\n⚠ WARNING: No legend colors loaded!")

    def normalize_identifier(self, size, code, number, suffix=None):
        """Normalize and fix OCR errors"""
        code = code.upper()

        # Fix OCR errors
        ocr_fixes = {
            '0W': 'OW', '0C': 'OC', 'C0': 'CO',
            'CQ': 'CG', 'NC': 'NG'
        }
        code = ocr_fixes.get(code, code)

        if not (len(code) == 2 and code.isalpha()):
            return None

        number = number.replace('O', '0').replace('I', '1')
        if not (4 <= len(number) <= 6 and number.isdigit()):
            return None

        if suffix:
            suffix = suffix.upper().replace('O', '0').replace('I', '1')
            if not (2 <= len(suffix) <= 4 and suffix[0].isalpha() and suffix[1:].isdigit()):
                return None
            return f'{size}"-{code}-{number}-{suffix}'

        return None

    def extract_text_from_predictions(self, predictions) -> List[Tuple[str, tuple]]:
        """Extract text with bboxes from Surya predictions"""
        results = []
        for line in predictions[0].text_lines:
            text = line.text.strip()
            if text:
                results.append((text, line.bbox))
        return results

    def extract_complete_identifiers(self, texts_with_bboxes: List[Tuple[str, tuple]]) -> Set[Tuple[str, tuple]]:
        """Extract complete ISO identifiers with their bboxes"""
        found = set()

        for text, bbox in texts_with_bboxes:
            match = self.complete_pattern.search(text)
            if match:
                size = match.group(1)
                code = match.group(2)
                number = match.group(3)
                suffix = match.group(4)

                formatted = self.normalize_identifier(size, code, number, suffix)
                if formatted:
                    found.add((formatted, tuple(bbox)))

        return found

    def discover_pages(self) -> Dict[str, List[Path]]:
        """Discover all pages and their patches"""
        print("\n" + "="*80)
        print("DISCOVERING PAGES IN FOLDER")
        print("="*80)
        
        all_patches = list(self.patch_folder.glob("*_r*_c*.png"))
        print(f"Total patch files found: {len(all_patches)}")
        
        if not all_patches:
            print("⚠ No patch files found")
            return {}
        
        pages = {}
        for patch_file in all_patches:
            if '_r' in patch_file.name:
                page_name = patch_file.name.split('_r')[0]
                if page_name not in pages:
                    pages[page_name] = []
                pages[page_name].append(patch_file)
        
        for page_name in pages:
            pages[page_name] = sorted(pages[page_name])
        
        print(f"\nDiscovered {len(pages)} page(s):")
        for page_name in sorted(pages.keys()):
            print(f"  • {page_name}: {len(pages[page_name])} patches")
        
        print("="*80 + "\n")
        
        return pages

    def process_single_patch(self, patch_path: Path) -> Dict:
        """
        Process a single patch with color-based filtering
        
        Returns:
            {
                'skipped': bool,
                'identifiers': List of (identifier, legend_match) tuples,
                'reason': str (if skipped)
            }
        """
        print(f"\nProcessing: {patch_path.name}")

        # Load patch
        patch = cv2.imread(str(patch_path))
        if patch is None:
            return {'skipped': True, 'identifiers': [], 'reason': 'Failed to load'}

        # STEP 1: Quick check for colored lines
        if not has_colored_lines(patch):
            print(f"  ⊘ SKIPPED - No colored lines detected")
            return {'skipped': True, 'identifiers': [], 'reason': 'No colored lines'}

        print(f"  ✓ Colored lines detected")

        # STEP 2: Extract colored line regions
        line_regions = extract_colored_line_regions(patch, self.legend_colors)
        
        if not line_regions:
            print(f"  ⊘ SKIPPED - No matching legend colors found")
            return {'skipped': True, 'identifiers': [], 'reason': 'No legend color matches'}

        print(f"  ✓ Found {len(line_regions)} colored line region(s):")
        for lr in line_regions:
            print(f"    - {lr['legend_match']} ({lr['pattern']}) at {lr['bbox']}")

        # STEP 3: Run OCR
        patch_rgb = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)
        patch_pil = Image.fromarray(patch_rgb)
        predictions = self.rec_predictor([patch_pil], det_predictor=self.det_predictor)
        texts_with_bboxes = self.extract_text_from_predictions(predictions)

        # STEP 4: Extract complete identifiers
        complete_ids_with_bboxes = self.extract_complete_identifiers(texts_with_bboxes)

        if not complete_ids_with_bboxes:
            print(f"  ⊘ No ISO identifiers found")
            return {'skipped': False, 'identifiers': [], 'reason': 'No identifiers detected'}

        # STEP 5: Filter identifiers that are near colored lines
        filtered_identifiers = []

        for identifier, id_bbox in complete_ids_with_bboxes:
            # Check proximity to any colored line
            for line_region in line_regions:
                line_bbox = line_region['bbox']
                
                if is_text_near_line(id_bbox, line_bbox, proximity_threshold=30):
                    legend_match = line_region['legend_match']
                    filtered_identifiers.append((identifier, legend_match))
                    print(f"  ✓ {identifier} → {legend_match}")
                    break  # Found a match, no need to check other lines

        if not filtered_identifiers:
            print(f"  ⊘ Identifiers found but none adjacent to colored lines")

        return {
            'skipped': False,
            'identifiers': filtered_identifiers,
            'reason': None
        }

    def extract_from_page(self, page_name: str, patch_files: List[Path]) -> dict:
        """Extract ISO identifiers from a single page with color filtering"""
        print("="*80)
        print(f"EXTRACTING FROM PAGE: {page_name}")
        print("="*80)

        print(f"\nTotal patches: {len(patch_files)}")

        results_by_legend = {}
        skipped_count = 0
        processed_count = 0

        for patch_path in patch_files:
            result = self.process_single_patch(patch_path)

            if result['skipped']:
                skipped_count += 1
            else:
                processed_count += 1

            # Group identifiers by legend
            for identifier, legend_match in result['identifiers']:
                if legend_match not in results_by_legend:
                    results_by_legend[legend_match] = set()
                results_by_legend[legend_match].add(identifier)

        # Convert sets to sorted lists
        for legend in results_by_legend:
            results_by_legend[legend] = sorted(results_by_legend[legend])

        total_identifiers = sum(len(ids) for ids in results_by_legend.values())

        return {
            'page_name': page_name,
            'total_patches': len(patch_files),
            'skipped_patches': skipped_count,
            'processed_patches': processed_count,
            'results_by_legend': results_by_legend,
            'total_identifiers': total_identifiers
        }

    def extract_from_all_pages(self) -> Dict[str, dict]:
        """Extract from all pages"""
        pages = self.discover_pages()
        
        if not pages:
            return {}
        
        all_results = {}
        
        for page_name in sorted(pages.keys()):
            patch_files = pages[page_name]
            results = self.extract_from_page(page_name, patch_files)
            all_results[page_name] = results
        
        return all_results


# ============================================================================
# MAIN
# ============================================================================

def main():
    # Configuration
    patch_folder = r"C:\Users\saroc\Desktop\P&ID V2\iso-identifier-test-patches\test-node1.4-patches"
    legend_path = r"C:\Users\saroc\Desktop\P&ID V2\legends-sheets\GBN-EPIC-ERCP_1.png"  # Update with your legend path

    # Create extractor
    extractor = ColorOptimizedPatchExtractor(patch_folder, legend_path)

    # Extract from all pages
    all_results = extractor.extract_from_all_pages()

    # Print results
    print("\n" + "="*80)
    print("FINAL RESULTS - COLOR-FILTERED ISO IDENTIFIERS")
    print("="*80)
    
    total_pages = len(all_results)
    total_patches = sum(r['total_patches'] for r in all_results.values())
    total_skipped = sum(r['skipped_patches'] for r in all_results.values())
    total_processed = sum(r['processed_patches'] for r in all_results.values())
    total_identifiers = sum(r['total_identifiers'] for r in all_results.values())
    
    print(f"\nTotal pages: {total_pages}")
    print(f"Total patches: {total_patches}")
    print(f"  - Skipped (no colored lines): {total_skipped}")
    print(f"  - Processed: {total_processed}")
    print(f"Total identifiers extracted: {total_identifiers}")
    
    # Print per-page results grouped by legend
    print("\n" + "-"*80)
    print("RESULTS BY LEGEND")
    print("-"*80)
    
    # Aggregate all results by legend across all pages
    global_legend_results = {}
    
    for page_name, page_results in all_results.items():
        for legend_name, identifiers in page_results['results_by_legend'].items():
            if legend_name not in global_legend_results:
                global_legend_results[legend_name] = []
            
            for identifier in identifiers:
                global_legend_results[legend_name].append({
                    'identifier': identifier,
                    'page': page_name
                })
    
    # Print grouped by legend
    for legend_name in sorted(global_legend_results.keys()):
        entries = global_legend_results[legend_name]
        color_info = extractor.legend_colors.get(legend_name, {})
        color_hex = color_info.get('hex', 'N/A')
        pattern = color_info.get('pattern', 'N/A')
        
        print(f"\n📌 {legend_name} ({color_hex}, {pattern}):")
        print(f"   Total identifiers: {len(entries)}")
        
        for entry in entries:
            print(f"     • {entry['identifier']} (page: {entry['page']})")
    
    print("\n" + "="*80)
    
    return all_results


if __name__ == "__main__":
    main()


✓ Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Initializing OCR...

Loading legend from: C:\Users\saroc\Desktop\P&ID V2\legends-sheets\GBN-EPIC-ERCP_1.png


Recognizing Text: 100%|██████████| 65/65 [01:21<00:00,  1.26s/it]


  Detected 65 text lines from legend

✓ Loaded 17 legend entries:
  • GBN-CPP-01           #f69786    (solid)
  • GBN-CPP-02           #fed47e    (solid)
  • GBN-CPP-03           #f59385    (dashed)
  • GBN-CPP-04           #d3df7d    (solid)
  • GBN-CPP-05           #88d6f6    (solid)
  • GBN-CPP-06           #d3df7d    (dashed)
  • GBN-CPP-07           #88d6f6    (dashed)
  • GBN-CPP-08           #fed47e    (solid)
  • GBN-CPP-09           #f7f17e    (solid)
  • GBN-CPP-10           #bc7eb6    (solid)
  • GBN-CPP-11           #dcd2e8    (solid)
  • GBN-CPP-12           #f7bed7    (solid)
  • GBN-CPP-13           #fcded3    (solid)
  • GBN-CPP-14           #f7bed7    (dashed)
  • GBN-CPP-15           #fed47e    (dashed)
  • GBN-CPP-16           #bc7fb6    (dashed)
  • GBN-CPP-22           #98d8e5    (dashed)

DISCOVERING PAGES IN FOLDER
Total patch files found: 64

Discovered 2 page(s):
  • ERCP_2: 24 patches
  • ERCP_3: 40 patches

EXTRACTING FROM PAGE: ERCP_2

Total patches: 24

Pro

Recognizing Text: 100%|██████████| 35/35 [00:07<00:00,  4.98it/s]


  ⊘ No ISO identifiers found

Processing: ERCP_2_r0_c2.png
  ✓ Colored lines detected
  ✓ Found 6 colored line region(s):
    - GBN-CPP-13 (dashed) at (np.int32(39), np.int32(114), np.int32(337), np.int32(142))
    - GBN-CPP-13 (dashed) at (np.int32(39), np.int32(149), np.int32(364), np.int32(178))
    - GBN-CPP-13 (dashed) at (np.int32(92), np.int32(244), np.int32(158), np.int32(306))
    - GBN-CPP-13 (dashed) at (np.int32(550), np.int32(245), np.int32(617), np.int32(308))
    - GBN-CPP-13 (dashed) at (np.int32(393), np.int32(270), np.int32(463), np.int32(296))
    - GBN-CPP-13 (dashed) at (np.int32(50), np.int32(296), np.int32(107), np.int32(323))


...

  ✓ 3"-CD-8311-D03 → GBN-CPP-09

Processing: ERCP_3_r4_c4.png
  ✓ Colored lines detected
  ✓ Found 5 colored line region(s):
    - GBN-CPP-09 (dashed) at (np.int32(0), np.int32(0), np.int32(640), np.int32(435))
    - GBN-CPP-02 (dashed) at (np.int32(479), np.int32(450), np.int32(523), np.int32(497))
    - GBN-CPP-02 (solid) at (np.int32(531), np.int32(451), np.int32(570), np.int32(496))
    - GBN-CPP-02 (dashed) at (np.int32(577), np.int32(451), np.int32(616), np.int32(496))
    - GBN-CPP-02 (solid) at (np.int32(621), np.int32(474), np.int32(640), np.int32(483))


Recognizing Text: 100%|██████████| 11/11 [00:01<00:00,  6.97it/s]


  ✓ 24"-NG-8186-D06 → GBN-CPP-09
  ✓ 2"-CL-8136-D48 → GBN-CPP-09

Processing: ERCP_3_r4_c5.png
  ✓ Colored lines detected
  ✓ Found 11 colored line region(s):
    - GBN-CPP-09 (solid) at (np.int32(0), np.int32(415), np.int32(168), np.int32(435))
    - GBN-CPP-02 (dashed) at (np.int32(0), np.int32(450), np.int32(43), np.int32(497))
    - GBN-CPP-02 (solid) at (np.int32(51), np.int32(451), np.int32(90), np.int32(496))
    - GBN-CPP-02 (dashed) at (np.int32(97), np.int32(451), np.int32(136), np.int32(496))
    - GBN-CPP-02 (solid) at (np.int32(164), np.int32(450), np.int32(206), np.int32(497))
    - GBN-CPP-02 (solid) at (np.int32(212), np.int32(451), np.int32(249), np.int32(496))
    - GBN-CPP-02 (solid) at (np.int32(256), np.int32(451), np.int32(292), np.int32(496))
    - GBN-CPP-02 (dashed) at (np.int32(317), np.int32(451), np.int32(349), np.int32(497))
    - GBN-CPP-02 (solid) at (np.int32(352), np.int32(451), np.int32(385), np.int32(497))
    - GBN-CPP-02 (solid) at (np.int32(141), n

Recognizing Text: 100%|██████████| 47/47 [00:06<00:00,  7.20it/s]


  ⊘ Identifiers found but none adjacent to colored lines

Processing: ERCP_3_r4_c6.png
  ⊘ SKIPPED - No colored lines detected

Processing: ERCP_3_r4_c7.png
  ⊘ SKIPPED - No colored lines detected

FINAL RESULTS - COLOR-FILTERED ISO IDENTIFIERS

Total pages: 2
Total patches: 64
  - Skipped (no colored lines): 11
  - Processed: 53
Total identifiers extracted: 31

--------------------------------------------------------------------------------
RESULTS BY LEGEND
--------------------------------------------------------------------------------

📌 GBN-CPP-02 (#fed47e, solid):
   Total identifiers: 12
     • 24"-NG-8114-D48 (page: ERCP_2)
     • 24"-NG-8831-D48 (page: ERCP_2)
     • 2"-NG-8134-D48 (page: ERCP_3)
     • 24"-NG-8131-D48 (page: ERCP_3)
     • 24"-NG-8132-D48 (page: ERCP_3)
     • 24"-NG-8139-D48 (page: ERCP_3)
     • 24"-NG-8141-D48 (page: ERCP_3)
     • 24"-NG-8191-D48 (page: ERCP_3)
     • 6"-NG-8133-D48 (page: ERCP_3)
     • 6"-NG-8136-D48 (page: ERCP_3)
     • 8"-HF-8514-B03